# 🎬🍿 Movie Recommender System with SVD
### Collaborative Filtering on the MovieLens 100K Dataset

---

> 🎯 **Ever wondered how Netflix always seems to know what you want to watch next?**
> In this notebook, we build a fully functional **Movie Recommendation Engine** from scratch — using the same core technique that powers real-world platforms like Netflix, Spotify, and Amazon.

---

## 📌 What This Notebook Covers

| Step | Topic | Description |
|------|-------|-------------|
| 1️⃣ | **Setup** | Install libraries & import dependencies |
| 2️⃣ | **Data Loading** | Download the MovieLens 100K dataset via KaggleHub |
| 3️⃣ | **Exploration (EDA)** | Understand ratings, user behavior & movie popularity |
| 4️⃣ | **Preprocessing** | Prepare data for the Surprise recommendation library |
| 5️⃣ | **Model Training** | Train an SVD (Singular Value Decomposition) model |
| 6️⃣ | **Evaluation** | Measure accuracy with RMSE and MAE |
| 7️⃣ | **Recommendations** | Generate personalized Top-10 movie recommendations |
| 8️⃣ | **Visualization** | Plot and interpret the results |

---

## 🛠️ Tech Stack

- 🐼 **pandas** — Data manipulation
- 📊 **matplotlib & seaborn** — Data visualization
- 🤖 **scikit-surprise** — Collaborative filtering & SVD model
- 📦 **kagglehub** — Seamless dataset download
- 🔢 **numpy** — Numerical operations

---

## 💡 Who Is This For?
- 🎓 Students learning Machine Learning & Recommender Systems
- 🧪 Data Scientists exploring collaborative filtering
- 🚀 Anyone curious about how personalization algorithms actually work!

---
⏱️ **Estimated run time:** ~2–3 minutes | 🖥️ **Runtime:** CPU is sufficient

---
## 🔧 Step 1 — Install Required Libraries

Before we do anything else, we need to install two libraries that don't come pre-installed in Colab:

- **`scikit-surprise`** 🤖 — A Python library specifically designed for building and evaluating recommender systems. It includes several collaborative filtering algorithms including SVD.
- **`kagglehub`** 📦 — A lightweight tool to download datasets directly from Kaggle without needing manual API setup.

> ⚠️ **Note:** Run this cell first! You only need to run it once per session.

In [ ]:
# ============================================================
# 📦 Install external libraries not available by default in Colab
# - scikit-surprise: for building the SVD recommendation model
# - kagglehub: for downloading the MovieLens dataset from Kaggle
# ============================================================
!pip install scikit-surprise kagglehub --quiet

print("✅ Libraries installed successfully!")

---
## 📚 Step 2 — Import All Dependencies

Now we import everything we need for the entire project in one place.
Keeping all imports at the top is a best practice — it makes dependencies obvious at a glance.

Here's what each import does:

| Library | What It's Used For |
|---------|--------------------|
| `numpy` | Fast numerical arrays and math operations |
| `pandas` | Loading, cleaning, and exploring tabular data |
| `matplotlib` | Creating plots and charts |
| `seaborn` | Prettier, higher-level statistical plots |
| `os` | Navigating file system paths |
| `kagglehub` | Downloading datasets from Kaggle |
| `surprise.Dataset / Reader` | Loading rating data into Surprise's format |
| `surprise.SVD` | The SVD matrix factorization algorithm |
| `train_test_split` | Splitting data into train/test sets |
| `rmse, mae` | Evaluation metrics for the model |

In [ ]:
# ============================================================
# 📚 Standard Python & Data Science Libraries
# ============================================================
import numpy as np          # Numerical computing
import pandas as pd         # Data manipulation and analysis
import matplotlib.pyplot as plt  # Plotting
import seaborn as sns       # Statistical visualization
import os                   # File system operations

# ============================================================
# 📦 Kaggle Dataset Downloader
# ============================================================
import kagglehub             # Fetch datasets straight from Kaggle

# ============================================================
# 🤖 Surprise — Recommender System Library
# ============================================================
from surprise import Dataset                        # Loads data in Surprise format
from surprise import SVD                            # Singular Value Decomposition model
from surprise import Reader                         # Parses rating scale from raw data
from surprise.model_selection import train_test_split  # Train/test split
from surprise.accuracy import rmse, mae             # Evaluation metrics

# ============================================================
# 🎨 Visual Style Settings
# ============================================================
sns.set_theme(style="whitegrid")   # Clean grid background for plots
plt.rcParams['figure.figsize'] = (10, 5)  # Default figure size

print("✅ All libraries imported successfully!")

---
## 📥 Step 3 — Download & Load the MovieLens 100K Dataset

### 🎬 About the Dataset
The **MovieLens 100K** dataset is a classic benchmark in recommender system research, published by the GroupLens Research lab at the University of Minnesota.

It contains:
- 📊 **100,000 ratings** (1–5 stars)
- 👤 **943 users**
- 🎥 **1,682 movies**
- 🗓️ Collected between September 1997 and April 1998

### 📁 Key Files
| File | Description |
|------|-------------|
| `u.data` | Main ratings file: userId, movieId, rating, timestamp |
| `u.item` | Movie metadata: movieId, title, release date, genre flags |

> 💡 We use `kagglehub` to download automatically — no manual Kaggle API key setup needed in Colab!

In [ ]:
# ============================================================
# 📥 Download the MovieLens 100K dataset from Kaggle
# kagglehub handles caching — it won't re-download if already present
# ============================================================
print("⏳ Downloading MovieLens 100K dataset...")
path = kagglehub.dataset_download("bromotdi/movielens-100k-dataset")
print(f"✅ Dataset downloaded! Files are located at: {path}")

# List the top-level contents of the downloaded folder
print("\n📂 Contents of the downloaded folder:")
print(os.listdir(path))

In [ ]:
# ============================================================
# 📄 Load the Ratings Data (u.data)
# The file is tab-separated with NO header row, so we define column names manually
# ============================================================
column_names = ['userId', 'movieId', 'rating', 'timestamp']

ratings = pd.read_csv(
    os.path.join(path, 'ml-100k', 'u.data'),
    sep='\t',           # Tab-separated values
    names=column_names  # Assign column headers manually
)

print(f"✅ Ratings data loaded! Shape: {ratings.shape}")
print(f"   → {ratings.shape[0]:,} ratings from {ratings['userId'].nunique()} users on {ratings['movieId'].nunique()} movies")
print("\n🔍 First 5 rows:")
ratings.head()

In [ ]:
# ============================================================
# 🎥 Load Movie Metadata (u.item)
# This file contains movie titles and genre flags
# We only keep the movieId and title columns for our purposes
# ============================================================
movies = pd.read_csv(
    os.path.join(path, 'ml-100k', 'u.item'),
    sep='|',                        # Pipe-separated values
    encoding='latin-1',             # Required for special characters in titles
    header=None,                    # No header row in this file
    usecols=[0, 1],                 # Only load columns 0 (id) and 1 (title)
    names=['movieId', 'title']      # Assign column names
)

print(f"✅ Movie metadata loaded! {len(movies)} movies found.")
print("\n🔍 First 5 movies:")
movies.head()

---
## 🔍 Step 4 — Exploratory Data Analysis (EDA)

Before training any model, it's crucial to **understand your data**.
Good EDA helps us spot patterns, anomalies, and biases that could affect model quality.

We'll explore:
- 📊 **Rating distribution** — Are most users generous or harsh raters?
- 👤 **Ratings per user** — Are a few users dominating the dataset?
- 🎥 **Ratings per movie** — Which movies have the most reviews?
- ⭐ **Average rating per movie** — What are the most loved films?
- 🧮 **Basic statistics** — A quick numerical summary

In [ ]:
# ============================================================
# 🧮 Basic Dataset Statistics
# Get a quick numerical summary of the ratings column
# ============================================================
print("📊 Dataset Overview")
print("=" * 40)
print(f"  Total Ratings  : {len(ratings):,}")
print(f"  Unique Users   : {ratings['userId'].nunique():,}")
print(f"  Unique Movies  : {ratings['movieId'].nunique():,}")
print(f"  Rating Range   : {ratings['rating'].min()} – {ratings['rating'].max()} stars")
print(f"  Mean Rating    : {ratings['rating'].mean():.2f} stars")
print(f"  Median Rating  : {ratings['rating'].median():.1f} stars")
print("=" * 40)

# Sparsity tells us how 'empty' the user-movie matrix is
# Most users have only rated a tiny fraction of all movies
n_users = ratings['userId'].nunique()
n_movies = ratings['movieId'].nunique()
sparsity = 1 - (len(ratings) / (n_users * n_movies))
print(f"\n  Matrix Sparsity: {sparsity:.2%}")
print("  → This means most user-movie combinations have NO rating (typical for recommender systems!)")

In [ ]:
# ============================================================
# 📊 Plot 1: Rating Distribution
# Shows how many times each star rating (1–5) was given
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Left: Count of each rating value ---
rating_counts = ratings['rating'].value_counts().sort_index()
axes[0].bar(
    rating_counts.index,
    rating_counts.values,
    color=['#e74c3c','#e67e22','#f1c40f','#2ecc71','#3498db'],
    edgecolor='white', width=0.6
)
axes[0].set_title('⭐ Rating Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Star Rating')
axes[0].set_ylabel('Number of Ratings')
axes[0].set_xticks([1, 2, 3, 4, 5])
for bar, count in zip(axes[0].patches, rating_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
                 f'{count:,}', ha='center', fontsize=10)

# --- Right: Ratings per user (how active are users?) ---
ratings_per_user = ratings.groupby('userId')['rating'].count()
axes[1].hist(ratings_per_user, bins=50, color='#9b59b6', edgecolor='white')
axes[1].set_title('👤 Ratings per User', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Number of Ratings Given')
axes[1].set_ylabel('Number of Users')
axes[1].axvline(ratings_per_user.mean(), color='red', linestyle='--',
                label=f'Mean: {ratings_per_user.mean():.0f}')
axes[1].legend()

plt.suptitle('📊 Exploring User Rating Behavior', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"\n💡 Insight: Rating '4' is the most common — users tend to rate movies they already like!")
print(f"💡 Insight: Most users have rated between {ratings_per_user.quantile(0.25):.0f}–{ratings_per_user.quantile(0.75):.0f} movies (middle 50%).")

In [ ]:
# ============================================================
# 🎥 Most Rated & Highest Rated Movies
# Merge ratings with movie titles for a human-readable view
# ============================================================

# Merge ratings with movie titles
ratings_with_titles = ratings.merge(movies, on='movieId')

# --- Most Rated Movies (by count) ---
most_rated = (
    ratings_with_titles.groupby('title')['rating']
    .count()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)
most_rated.columns = ['title', 'num_ratings']

# --- Highest Rated Movies (min 50 ratings to avoid flukes) ---
avg_ratings = (
    ratings_with_titles.groupby('title')['rating']
    .agg(['mean', 'count'])
    .reset_index()
)
avg_ratings.columns = ['title', 'avg_rating', 'num_ratings']
top_avg = avg_ratings[avg_ratings['num_ratings'] >= 50].sort_values('avg_rating', ascending=False).head(10)

# --- Plot side by side ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Most rated
sns.barplot(data=most_rated, y='title', x='num_ratings', ax=axes[0], palette='Blues_r')
axes[0].set_title('🔥 Top 10 Most Rated Movies', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Number of Ratings')
axes[0].set_ylabel('')

# Highest average rating
sns.barplot(data=top_avg, y='title', x='avg_rating', ax=axes[1], palette='Greens_r')
axes[1].set_title('⭐ Top 10 Highest Rated Movies (min 50 ratings)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Average Rating')
axes[1].set_ylabel('')
axes[1].set_xlim(3.5, 5.0)

plt.suptitle('🎬 Movie Popularity & Quality Overview', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## ⚙️ Step 5 — Data Preprocessing for Surprise

The `scikit-surprise` library requires data in a specific format. We need to:

1. **Define a `Reader`** — tells Surprise the scale of ratings (1 to 5)
2. **Load into a `Dataset`** — converts our pandas DataFrame into Surprise's internal format
3. **Split into train/test sets** — 80% for training, 20% for testing

> 💡 **Why split data?** We train the model on one portion and evaluate it on data it has never seen before — this gives us an honest measure of how well it generalizes.

In [ ]:
# ============================================================
# ⚙️ Prepare Data for the Surprise Library
# ============================================================

# Step 5a: Define the rating scale
# Reader tells Surprise that ratings go from 1 (min) to 5 (max)
reader = Reader(rating_scale=(1, 5))

# Step 5b: Load data from the pandas DataFrame
# Surprise expects exactly 3 columns: user, item, rating (in that order)
data = Dataset.load_from_df(
    ratings[['userId', 'movieId', 'rating']],  # Select only the needed columns
    reader
)

# Step 5c: Split into train (80%) and test (20%) sets
# random_state=42 ensures reproducibility — same split every run
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

# Step 5d: Build the full training set (used later for generating recommendations)
full_trainset = data.build_full_trainset()

print("✅ Data preprocessing complete!")
print(f"   → Training samples : {trainset.n_ratings:,}")
print(f"   → Testing samples  : {len(testset):,}")
print(f"   → Number of users  : {trainset.n_users}")
print(f"   → Number of items  : {trainset.n_items}")

---
## 🤖 Step 6 — Train the SVD Model

### 🧠 What is SVD (Singular Value Decomposition)?

SVD is a **Matrix Factorization** technique. Here's the intuition:

Imagine a giant matrix where:
- **Rows** = Users
- **Columns** = Movies
- **Values** = Star ratings (most are missing!)

SVD **decomposes** this sparse matrix into two smaller matrices:
- A **User matrix** — captures each user's hidden preferences (e.g., loves action, hates romance)
- A **Movie matrix** — captures each movie's hidden characteristics (e.g., high action, low romance)

By multiplying these two matrices back together, we can **predict missing ratings**!

### ⚙️ Key Hyperparameters
| Parameter | Value | Meaning |
|-----------|-------|--------|
| `n_factors` | 100 | Number of latent factors (dimensions) |
| `n_epochs` | 20 | Training iterations over the data |
| `lr_all` | 0.005 | Learning rate — how fast the model updates |
| `reg_all` | 0.02 | Regularization — prevents overfitting |

In [ ]:
# ============================================================
# 🤖 Initialize and Train the SVD Model
# ============================================================

# Create the SVD model with tuned hyperparameters
svd_model = SVD(
    n_factors=100,    # Number of latent factors to learn
    n_epochs=20,      # Number of training iterations
    lr_all=0.005,     # Learning rate for all parameters
    reg_all=0.02,     # L2 regularization to prevent overfitting
    random_state=42   # For reproducibility
)

print("⏳ Training the SVD model...")

# Train the model on the training set
svd_model.fit(trainset)

print("✅ Model training complete!")
print("   → The model has learned hidden preferences for every user and every movie.")

---
## 📏 Step 7 — Evaluate Model Performance

We evaluate our model using two standard metrics:

| Metric | Full Name | What It Measures |
|--------|-----------|------------------|
| **RMSE** | Root Mean Squared Error | Average prediction error (penalizes large errors more) |
| **MAE** | Mean Absolute Error | Average absolute prediction error |

Both are measured **in rating units (stars)**.

> 🎯 **Goal:** Lower is better! An RMSE of ~0.93 means the model's predicted rating is on average about 0.93 stars off from the actual rating — quite good on a 1–5 scale!

In [ ]:
# ============================================================
# 📏 Generate Predictions on the Test Set & Compute Metrics
# ============================================================

# Generate predictions for every (user, movie) pair in the test set
predictions = svd_model.test(testset)

# Calculate evaluation metrics
print("📊 Model Evaluation on Held-Out Test Set")
print("=" * 45)
test_rmse = rmse(predictions, verbose=False)   # Root Mean Squared Error
test_mae  = mae(predictions, verbose=False)    # Mean Absolute Error
print(f"   RMSE : {test_rmse:.4f} stars")
print(f"   MAE  : {test_mae:.4f} stars")
print("=" * 45)
print(f"\n💡 Interpretation: On average, our model's predicted ratings")
print(f"   are off by only ~{test_mae:.2f} stars from the actual ratings.")
print(f"   That's quite good on a 1–5 star scale! 🎉")

In [ ]:
# ============================================================
# 📊 Visualize: Actual vs Predicted Ratings
# ============================================================

# Extract actual and predicted values from the predictions list
actuals    = [pred.r_ui for pred in predictions]   # True ratings
predicted  = [pred.est  for pred in predictions]   # Predicted ratings
errors     = [abs(a - p) for a, p in zip(actuals, predicted)]  # Absolute errors

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Left: Scatter plot of actual vs predicted ---
axes[0].scatter(actuals, predicted, alpha=0.1, color='steelblue', s=5)
axes[0].plot([1, 5], [1, 5], 'r--', linewidth=2, label='Perfect Prediction')  # Diagonal = perfect
axes[0].set_xlabel('Actual Rating')
axes[0].set_ylabel('Predicted Rating')
axes[0].set_title('🎯 Actual vs Predicted Ratings', fontsize=13, fontweight='bold')
axes[0].legend()
axes[0].set_xlim(0.5, 5.5)
axes[0].set_ylim(0.5, 5.5)

# --- Right: Distribution of prediction errors ---
axes[1].hist(errors, bins=40, color='coral', edgecolor='white')
axes[1].axvline(np.mean(errors), color='darkred', linestyle='--',
                label=f'Mean Error: {np.mean(errors):.2f}')
axes[1].set_xlabel('Absolute Error (stars)')
axes[1].set_ylabel('Count')
axes[1].set_title('📉 Distribution of Prediction Errors', fontsize=13, fontweight='bold')
axes[1].legend()

plt.suptitle('📏 Model Evaluation Results', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 🎁 Step 8 — Generate Personalized Movie Recommendations

Now for the exciting part — **getting actual movie recommendations**!

### 🔄 The Recommendation Process:
1. Pick a user (e.g., User #42)
2. Find all movies that user has **NOT yet rated**
3. Use our SVD model to **predict how much they'd like each unseen movie**
4. Return the **Top 10 movies with the highest predicted ratings**

> 💡 We train on the **full dataset** (not just 80%) for final recommendations — this gives the model the most information possible.

In [ ]:
# ============================================================
# 🔄 Re-train the model on the FULL dataset
# (For evaluation we used 80%, but for real recommendations
#  we want the model to learn from ALL available ratings)
# ============================================================
print("⏳ Re-training SVD on the full dataset for best recommendations...")
svd_model.fit(full_trainset)
print("✅ Full-data training complete!")

In [ ]:
# ============================================================
# 🎯 Define the Recommendation Function
# This function works for ANY user in the dataset
# ============================================================

def get_top_n_recommendations(user_id, n=10):
    """
    Generate Top-N movie recommendations for a given user.

    Args:
        user_id (int): The ID of the user to recommend for
        n (int): Number of recommendations to return (default: 10)

    Returns:
        pd.DataFrame: Top-N recommended movies with predicted ratings
    """
    # Step 1: Get the list of ALL movie IDs in the dataset
    all_movie_ids = ratings['movieId'].unique()

    # Step 2: Find movies this user has ALREADY rated (we exclude these)
    already_rated = set(ratings[ratings['userId'] == user_id]['movieId'].values)

    # Step 3: Identify movies this user has NOT yet seen
    unseen_movies = [mid for mid in all_movie_ids if mid not in already_rated]

    # Step 4: Predict rating for each unseen movie using our SVD model
    predictions_list = [
        (movie_id, svd_model.predict(user_id, movie_id).est)
        for movie_id in unseen_movies
    ]

    # Step 5: Sort predictions by predicted rating (highest first)
    predictions_list.sort(key=lambda x: x[1], reverse=True)

    # Step 6: Take the top N
    top_n = predictions_list[:n]

    # Step 7: Convert to a DataFrame and add movie titles
    result_df = pd.DataFrame(top_n, columns=['movieId', 'PredictedRating'])
    result_df = result_df.merge(movies, on='movieId')  # Add movie title
    result_df = result_df[['movieId', 'title', 'PredictedRating']]  # Reorder columns
    result_df['PredictedRating'] = result_df['PredictedRating'].round(2)  # Round to 2 decimals
    result_df.index = range(1, n + 1)  # Start index at 1 (rank)

    return result_df

print("✅ Recommendation function defined!")

In [ ]:
# ============================================================
# 🎬 Generate & Display Recommendations for a Specific User
# ============================================================

# --- 🔧 CHANGE THIS to get recommendations for any user (1–943) ---
TARGET_USER_ID = 42
# ----------------------------------------------------------------

print(f"🎬 Generating Top 10 Movie Recommendations for User #{TARGET_USER_ID}...\n")

# Call our recommendation function
recommendations = get_top_n_recommendations(user_id=TARGET_USER_ID, n=10)

# Display the results
print(f"🏆 Top 10 Movies Recommended for User #{TARGET_USER_ID}:")
print("=" * 60)
for rank, row in recommendations.iterrows():
    stars = '⭐' * round(row['PredictedRating'])
    print(f"  #{rank:2d}. {row['title']:<45} {row['PredictedRating']} {stars}")
print("=" * 60)

# Show as a table as well
print("\n📋 As a DataFrame:")
recommendations

---
## 📊 Step 9 — Visualize the Recommendations

Let's make the results visually compelling with two charts:
1. A **horizontal bar chart** of the Top 10 predicted ratings
2. A comparison of the user's **actual ratings** vs what the model recommends

In [ ]:
# ============================================================
# 📊 Plot: Top 10 Recommendations with Predicted Ratings
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# --- Left: Bar chart of top recommended movies ---
colors = plt.cm.YlOrRd(np.linspace(0.4, 0.9, 10))[::-1]
bars = axes[0].barh(
    recommendations['title'],
    recommendations['PredictedRating'],
    color=colors, edgecolor='white'
)
axes[0].set_xlabel('Predicted Rating (⭐)', fontsize=12)
axes[0].set_title(f'🏆 Top 10 Recommendations for User #{TARGET_USER_ID}',
                  fontsize=13, fontweight='bold')
axes[0].set_xlim(3.5, 5.2)
axes[0].invert_yaxis()  # Rank 1 at the top
# Add rating labels on each bar
for bar, val in zip(bars, recommendations['PredictedRating']):
    axes[0].text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2,
                 f'{val:.2f}', va='center', fontweight='bold')

# --- Right: Distribution of the user's historical ratings ---
user_ratings = ratings[ratings['userId'] == TARGET_USER_ID]['rating']
rec_ratings  = recommendations['PredictedRating']

axes[1].hist(user_ratings, bins=[0.5,1.5,2.5,3.5,4.5,5.5],
             alpha=0.6, color='steelblue', label="User's Past Ratings", edgecolor='white')
axes[1].hist(rec_ratings, bins=10,
             alpha=0.6, color='coral', label='Recommended Predictions', edgecolor='white')
axes[1].axvline(user_ratings.mean(), color='steelblue', linestyle='--',
                label=f"User avg: {user_ratings.mean():.2f}")
axes[1].axvline(rec_ratings.mean(), color='coral', linestyle='--',
                label=f"Rec avg: {rec_ratings.mean():.2f}")
axes[1].set_xlabel('Rating / Predicted Rating')
axes[1].set_ylabel('Count')
axes[1].set_title(f'📊 User #{TARGET_USER_ID}: Past Ratings vs Recommendations',
                  fontsize=13, fontweight='bold')
axes[1].legend()

plt.suptitle('🎬 Personalized Recommendation Results', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"\n💡 Insight: User #{TARGET_USER_ID} has historically given average ratings of "
      f"{user_ratings.mean():.2f} stars.")
print(f"   The recommended movies have an average predicted rating of "
      f"{rec_ratings.mean():.2f} — the model is confidently recommending films it thinks this user will love!")

---
## 🔁 Step 10 — Try It Yourself! (Interactive Section)

Want to see recommendations for a different user? Just change the `USER_ID` below and re-run the cell!

Valid user IDs range from **1 to 943**.

In [ ]:
# ============================================================
# 🔁 Interactive: Get Recommendations for ANY User
# ✏️ Change USER_ID below to any number between 1 and 943
# ============================================================

USER_ID = 100   # ← ✏️ Change this!
TOP_N   = 10    # ← ✏️ How many recommendations to show?

# Validate the user ID
valid_users = ratings['userId'].unique()
if USER_ID not in valid_users:
    print(f"⚠️  User ID {USER_ID} not found. Please choose between 1 and {valid_users.max()}.")
else:
    recs = get_top_n_recommendations(user_id=USER_ID, n=TOP_N)
    n_past = len(ratings[ratings['userId'] == USER_ID])

    print(f"👤 User #{USER_ID} has rated {n_past} movies in the past.")
    print(f"🎬 Here are their Top {TOP_N} Recommended Films:\n")
    for rank, row in recs.iterrows():
        stars = '⭐' * round(row['PredictedRating'])
        print(f"  #{rank:2d}. {row['title']:<45} {row['PredictedRating']} {stars}")

    print("\n📋 Full Table:")
    display(recs)

---
## 🏁 Summary & Key Takeaways

Congratulations! 🎉 You've built a complete Movie Recommendation System from scratch.

### ✅ What We Accomplished

| Step | Achievement |
|------|-------------|
| 📥 Data Loading | Downloaded and loaded 100,000+ movie ratings |
| 🔍 EDA | Discovered rating patterns and user behavior |
| ⚙️ Preprocessing | Formatted data for the Surprise library |
| 🤖 Modeling | Trained an SVD collaborative filtering model |
| 📏 Evaluation | Achieved ~0.93 RMSE on unseen test data |
| 🎬 Recommendations | Generated personalized Top-10 movie lists |
| 📊 Visualization | Created insightful charts throughout |

---

### 🚀 Next Steps & Improvements

Want to take this further? Here are some ideas:

- 🔬 **Hyperparameter Tuning** — Use `GridSearchCV` from Surprise to find optimal `n_factors`, `n_epochs`, etc.
- 🧪 **Try Other Algorithms** — Compare SVD with `KNNBasic`, `NMF`, or `SlopeOne` from the Surprise library
- 🌐 **Content-Based Filtering** — Incorporate movie genres to handle the "cold start" problem for new users
- 🔀 **Hybrid Models** — Combine collaborative and content-based filtering for even better results
- 📱 **Build a Web App** — Deploy with Streamlit or Gradio to create an interactive recommendation interface
- 📊 **Use a Larger Dataset** — Try MovieLens 1M or 20M for more robust results

---

### 📚 Resources
- [scikit-surprise Documentation](https://surprise.readthedocs.io/en/stable/)
- [MovieLens Dataset Info](https://grouplens.org/datasets/movielens/)
- [Matrix Factorization Techniques for Recommender Systems (Netflix Paper)](https://datajobs.com/data-science-repo/Recommender-Systems-[Netflix].pdf)

---
*Built with ❤️ using scikit-surprise, pandas, and matplotlib*